<a href="https://colab.research.google.com/github/SantiagoAmaya99/Crack-Detector-Using-Deep-Learning-Models/blob/main/Prueba_de_modelos_Detecci%C3%B3n_de_grietas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Dataset_IA2'

Mounted at /content/drive
/content/drive/MyDrive/Dataset_IA2


In [ ]:
#@title Librerias
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import os, re, shutil
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
!cp "/content/drive/MyDrive/Dataset_IA2_C.zip" "/content/"

In [ ]:
!unzip -q "/content/Dataset_IA2_C.zip" -d "/content/dataset_crudo"

In [ ]:
RUTA_CRUDO = Path('/content/dataset_crudo')

def id_foto(nombre_archivo):
    m = re.search(r'(\d+)[-_]\d+', Path(nombre_archivo).stem)
    return m.group(1) if m else Path(nombre_archivo).stem

registros = []

for material in ['Decks', 'Pavements', 'Walls']:
    for clase, etiqueta in [('Cracked', 1), ('Non-cracked', 0)]:
        carpeta_origen = RUTA_CRUDO / material / clase

        if carpeta_origen.exists():
            archivos = [
                f for f in carpeta_origen.iterdir()
                if f.is_file()
                and f.suffix.lower() in ['.jpg', '.jpeg', '.png']
                and f.stat().st_size > 0
            ]

            for f in tqdm(archivos, desc=f'{material}/{clase}'):
                registros.append({
                    'ruta': str(f),
                    'etiqueta': etiqueta,
                    'grupo': f'{material}_{id_foto(f.name)}'
                })

df = pd.DataFrame(registros)
print(f"Total de imágenes válidas mapeadas: {len(df)}")

Walls/Non-cracked: 100%|██████████| 1475/1475 [00:00<00:00, 154117.84it/s]

Total de imágenes mapeadas correctamente: 43280


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

seed = 42

def dividir_por_grupo(datos, fraccion, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=fraccion, random_state=seed)
    idx_a, idx_b = next(gss.split(datos, groups=datos['grupo']))
    return datos.iloc[idx_a].reset_index(drop=True), datos.iloc[idx_b].reset_index(drop=True)

df_trainval, df_test = dividir_por_grupo(df, 0.15, seed)
df_train, df_val = dividir_por_grupo(df_trainval, 0.15 / 0.85, seed)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

Train: 30128 | Val: 7016 | Test: 6136


In [ ]:
import tensorflow as tf
from tensorflow import keras

seed = 42


TAM_IMG = (128, 128)
TAM_LOTE = 64
AUTOTUNE = tf.data.AUTOTUNE

n_total = len(df_train)
n1 = int(df_train.etiqueta.sum())
n0 = n_total - n1
PESO_CLASE = {0: n_total / (2 * n0), 1: n_total / (2 * n1)}

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1, fill_mode='reflect'),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
])

def cargar_imagen(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, TAM_IMG)
    return img, tf.cast(etiqueta, tf.float32)

def crear_dataset(datos, entrenamiento=False):
    rutas = datos['ruta'].astype(str).values
    etiquetas = datos['etiqueta'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))

    if entrenamiento:
        ds = ds.shuffle(len(datos), seed, reshuffle_each_iteration=True)

    ds = ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)

    if entrenamiento:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
        w0, w1 = PESO_CLASE[0], PESO_CLASE[1]
        ds = ds.map(lambda x, y: (x, y, tf.where(y > 0.5, w1, w0)), num_parallel_calls=AUTOTUNE)

    return ds.batch(TAM_LOTE).prefetch(AUTOTUNE)

ds_train = crear_dataset(df_train, entrenamiento=True)
ds_val = crear_dataset(df_val)
ds_test = crear_dataset(df_test)

In [ ]:
import tensorflow as tf

def obtener_callbacks(nombre_modelo):

    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f'{nombre_modelo}_feature_extraction.keras',
            monitor='val_auc',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]

#ResNET 50

In [ ]:
def build_resnet50_base(input_shape=(128, 128, 3)):
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False

    modelResNET = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])

    return modelResNET, base_model

modelResNET, base_model_resnet = build_resnet50_base()

modelResNET.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_resnet = modelResNET.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('resnet50')
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 23,536,641 (89.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

#EfficientNetB0

In [ ]:
def build_efficientnet_base(input_shape=(128, 128, 3)):
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False

    model_efficient = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])

    return model_efficient, base_model

model_efficient, base_model_efficient = build_efficientnet_base()

model_efficient.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_efficientnet = model_efficient.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('efficientnet_b0')
)

#VGG16

In [ ]:
def build_vgg16_base(input_shape=(128, 128, 3)):
    base_vgg = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_vgg.trainable = False

    model_vgg = models.Sequential([
        base_vgg,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])

    return model_vgg, base_vgg

model_vgg, base_vgg = build_vgg16_base()

model_vgg.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_vgg = model_vgg.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('vgg16')
)

#DenseNet121

In [ ]:
# 1. Base y arquitectura
base_dense = DenseNet121(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

base_dense.trainable = False

model_dense = models.Sequential([
    base_dense,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

# 3. Compilación (LR = 1e-4)
model_dense.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model_dense.summary()


In [ ]:
history_dense = model_dense.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('densenet121')
)

#ConvNeXt

In [ ]:
base_convnext = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_convnext.trainable = False

model_convnext = models.Sequential([
    base_convnext,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model_convnext.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_convnext = model_convnext.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('convnext_tiny')
)

#MobileNetV2

In [ ]:
def build_mobilenetv2_base(input_shape=(128, 128, 3)):
    base_mobilenet = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_mobilenet.trainable = False

    model_mobilenet = models.Sequential([
        base_mobilenet,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])

    return model_mobilenet, base_mobilenet

model_mobilenet, base_mobilenet = build_mobilenetv2_base()

model_mobilenet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_mobilenet = model_mobilenet.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('mobilenetv2')
)